<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_03_statistical_experiments_and_significance_testing/03_statistical_experiments_and_significance_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 1 folder
%cd notebooks/chapter_03_statistical_experiments_and_significance_testing

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 160 (delta 97), reused 71 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 556.93 KiB | 7.63 MiB/s, done.
Resolving deltas: 100% (97/97), done.
/content/Practical-Statistics-for-DS
[Errno 2] No such file or directory: 'notebooks/chapter_03_statistical_experiments_and_significance_testing'
/content/Practical-Statistics-for-DS


# Chapter 03: Statistical Experiments and Significance Testing

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 3: Statistical Experiments and Significance Testing

## Goals

In this notebook, I will learn:
- Statistical experiments and A/B testing
- Null hypothesis (H₀) and alternative hypothesis (H₁)
- Statistical significance
- P-values and their interpretation
- Permutation tests (resampling)
- T-tests (classical and Welch's)
- Type I and Type II errors
- Statistical power and sample size
- Multiple testing problem and corrections
- ANOVA and Chi-Square tests
- Multi-Arm Bandit algorithms
- P-hacking and its dangers

---

## 🎯 Learning Objectives

1. Design and interpret A/B tests with proper control groups
2. Formulate null and alternative hypotheses
3. Implement permutation tests to assess significance without distributional assumptions
4. Correctly interpret p-values and recognize their limitations
5. Mitigate multiple testing (alpha inflation) using FDR and Bonferroni corrections
6. Apply ANOVA and Chi-Square tests for group comparisons
7. Understand Multi-Arm Bandit algorithms (Epsilon-Greedy) for sequential testing
8. Simulate statistical power and required sample sizes

---

## 🤖 ML Relevance

> Why does experimental design matter for Machine Learning?
> * **Model Evaluation**: Comparing Model A vs. Model B on a test set is essentially an A/B test. Is the 0.5% AUC lift real, or just noise?
> * **Feature Selection**: Testing 1,000 features for correlation with the target is a massive multiple testing problem.
> * **Bandit Algorithms**: Used in recommendation systems and hyperparameter tuning.
> * **Resampling**: Permutation and bootstrap logic underpin cross-validation and ensemble methods.

---

## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy import stats
    from scipy.stats import ttest_ind, f_oneway, chi2_contingency, fisher_exact
    from statsmodels.stats.multitest import multipletests
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

print("Notebook setup complete")
```
</details>


---

## Data Preparation

<details>
<summary>Click to view solution</summary>

```python
# Simulate datasets used in the book
np.random.seed(42)

# 1. Web Page Session Times (Continuous - for permutation/t-tests)
n_a, n_b = 21, 15
time_a = np.random.normal(loc=120, scale=40, size=n_a)
time_b = np.random.normal(loc=145, scale=45, size=n_b)
time_b = np.maximum(time_b, 10)
time_a = np.maximum(time_a, 10)

df_sessions = pd.DataFrame({
    'session_time': np.concatenate([time_a, time_b]),
    'page': ['Page_A']*n_a + ['Page_B']*n_b
})

# 2. E-commerce Price Test (Conversion Rates)
n_price_a, n_price_b = 2000, 2000
conv_a = np.random.binomial(1, 0.045, n_price_a)
conv_b = np.random.binomial(1, 0.055, n_price_b)

df_price = pd.DataFrame({
    'converted': np.concatenate([conv_a, conv_b]),
    'price_group': ['Standard']*n_price_a + ['Discount']*n_price_b
})

# 3. Four Web Pages (for ANOVA)
df_four_pages = pd.DataFrame({
    'time': np.concatenate([
        np.random.normal(110, 30, 50),
        np.random.normal(125, 35, 50),
        np.random.normal(115, 32, 50),
        np.random.normal(140, 40, 50)
    ]),
    'page': ['P1']*50 + ['P2']*50 + ['P3']*50 + ['P4']*50
})

# 4. Simple A/B test data
group_a = np.random.binomial(1, 0.12, 1000)
group_b = np.random.binomial(1, 0.14, 1000)

print("Datasets created successfully.")
print(f"Sessions: {df_sessions.shape}, Price Test: {df_price.shape}, Four Pages: {df_four_pages.shape}")
```
</details>


---

# Section 1: Statistical Experiments

## What is an experiment?

A statistical experiment compares different treatments to determine whether changes matter.

Example:
- **Group A**: Current website
- **Group B**: New website

Question: Did the new website improve conversions?

<details>
<summary>Click to view solution</summary>

```python
print("Group A Conversion Rate:", f"{group_a.mean():.3f}")
print("Group B Conversion Rate:", f"{group_b.mean():.3f}")

comparison = pd.DataFrame({
    "Group": ["A", "B"],
    "Conversion Rate": [group_a.mean(), group_b.mean()]
})
print(comparison)

plt.figure(figsize=(7, 5))
sns.barplot(data=comparison, x="Group", y="Conversion Rate")
plt.title("A/B Test Conversion Rates")
plt.show()
```
</details>

## Reflection

Questions:
1. Does Group B look better?
2. Is the difference meaningful?
3. Could randomness explain this?


---

# Section 2: Hypothesis Testing

## What is hypothesis testing?

Hypothesis testing helps answer: Is observed difference real, or did randomness create it?

### Null Hypothesis (H₀)
Default assumption: No difference exists.
Example: New website has no effect.

### Alternative Hypothesis (H₁)
Competing assumption: Difference exists.
Example: New website improves conversions.

### Key idea
We do NOT prove hypotheses. We only reject H₀ or fail to reject H₀.



---

# Section 3: A/B Testing and Initial Exploration

## Initial Look at the Session Time Data

Before testing, always visualize and summarize the data.

<details>
<summary>Click to view solution</summary>

```python
summary = df_sessions.groupby('page')['session_time'].agg(['count', 'mean', 'median', 'std'])
display(summary)

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_sessions, x='page', y='session_time', palette='Set2')
sns.stripplot(data=df_sessions, x='page', y='session_time', color='black', alpha=0.5, size=4, jitter=True)
plt.title('Session Times by Page Layout')
plt.ylabel('Time (seconds)')
plt.grid(axis='y', alpha=0.3)
plt.show()

obs_diff = summary.loc['Page_B', 'mean'] - summary.loc['Page_A', 'mean']
print(f"Observed Difference in Means (B - A): {obs_diff:.2f} seconds")
```
</details>


---

# Section 4: Statistical Significance

## What is statistical significance?

Statistical significance asks: Would this result happen by random chance?

Common threshold: α = 0.05

If p < 0.05, we often call result statistically significant.

⚠️ Important: Statistical significance does NOT mean business importance. Tiny effects can still become significant with enough data.


---

# Section 5: P-value Intuition

## What is a p-value?

P-value measures: How surprising our result is, assuming H₀ is true.

- Small p-value: Evidence against H₀
- Large p-value: Weak evidence against H₀

### Common misconceptions
P-value is NOT:
- Probability H₀ is true
- Probability result happened by chance

<details>
<summary>Click to view solution</summary>

```python
t_stat, p_value = stats.ttest_ind(group_a, group_b)

print("T-statistic:", t_stat)
print("P-value:", p_value)

alpha = 0.05
if p_value < alpha:
    print("Reject Null Hypothesis - Result is statistically significant")
else:
    print("Fail to Reject Null Hypothesis")
```
</details>

## Reflection

Questions:
1. Was the result significant?
2. Does significance guarantee usefulness?
3. Why can large datasets detect tiny effects?



---

# Section 6: The Permutation Test

## What is a permutation test?

Permutation testing asks: What if group labels were random?

If there is truly no difference between groups, then randomly shuffling labels should produce similar results.

### Advantages
- Fewer assumptions
- Intuitive
- Works well in A/B testing
- Robust to non-normal data

### Permutation Test Algorithm
1. Combine all data from both groups into one pool
2. Shuffle the pooled data randomly
3. Deal the shuffled data back into two groups of original sizes
4. Calculate the difference in means for this shuffled deal
5. Repeat steps 2-4 many times (e.g., 10,000 times)
6. See where your actual observed difference falls in this null distribution

<details>
<summary>Click to view solution</summary>

```python
def permutation_test(group_a, group_b, n_permutations=10000, seed=42):
    """
    Performs a two-sided permutation test for the difference in means.
    """
    np.random.seed(seed)
    
    # Observed statistic
    obs_diff = np.mean(group_b) - np.mean(group_a)
    
    # Pool the data
    pooled = np.concatenate([group_a, group_b])
    n_a = len(group_a)
    
    # Permutation loop
    perm_diffs = np.zeros(n_permutations)
    for i in range(n_permutations):
        np.random.shuffle(pooled)
        perm_a = pooled[:n_a]
        perm_b = pooled[n_a:]
        perm_diffs[i] = np.mean(perm_b) - np.mean(perm_a)
        
    # Calculate p-value (two-sided)
    p_value = np.mean(np.abs(perm_diffs) >= np.abs(obs_diff))
    
    return obs_diff, perm_diffs, p_value

# Run the test on session data
time_a = df_sessions[df_sessions['page'] == 'Page_A']['session_time'].values
time_b = df_sessions[df_sessions['page'] == 'Page_B']['session_time'].values

obs_diff, perm_diffs, p_val = permutation_test(time_a, time_b, n_permutations=10000)

print(f"Observed Difference: {obs_diff:.2f}s")
print(f"Permutation p-value: {p_val:.4f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Visualize the null distribution
plt.figure(figsize=(10, 5))

plt.hist(perm_diffs, bins=50, density=True, alpha=0.7, color='skyblue',
         edgecolor='black', label='Permutation Null Distribution')

plt.axvline(obs_diff, color='red', linestyle='--', linewidth=2,
            label=f'Observed Diff ({obs_diff:.2f}s)')
plt.axvline(-obs_diff, color='red', linestyle='--', linewidth=2, alpha=0.5)

plt.xlabel('Difference in Means (Page B - Page A)')
plt.ylabel('Density')
plt.title(f'Permutation Test Null Distribution (p = {p_val:.4f})')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

if p_val < 0.05:
    print("Conclusion: Reject the null hypothesis at alpha=0.05. The difference is statistically significant.")
else:
    print("Conclusion: Fail to reject the null hypothesis. The difference could be due to chance.")
```
</details>

## Simple Permutation Test

<details>
<summary>Click to view solution</summary>

```python
# Simpler permutation test with A/B test data
observed_difference = group_b.mean() - group_a.mean()
print("Observed Difference:", observed_difference)

combined = np.concatenate([group_a, group_b])
permutation_differences = []

for _ in range(1000):
    shuffled = np.random.permutation(combined)
    new_a = shuffled[:1000]
    new_b = shuffled[1000:]
    diff = new_b.mean() - new_a.mean()
    permutation_differences.append(diff)

plt.figure(figsize=(10, 6))
sns.histplot(permutation_differences, bins=30, kde=True)
plt.axvline(observed_difference, linestyle="--", label="Observed Difference")
plt.legend()
plt.title("Permutation Test Distribution")
plt.show()

p_value_perm = np.mean(np.abs(permutation_differences) >= np.abs(observed_difference))
print("Permutation P-value:", p_value_perm)
```
</details>

## Reflection

Questions:
1. Was observed difference unusual?
2. Why does shuffling help?
3. Why is permutation testing powerful?


---

# Section 7: T-tests

## What is a t-test?

A t-test compares group means and determines whether differences are likely due to randomness or real effects.

### Types of t-tests

**Independent t-test**: Used when groups are separate (e.g., Website A vs Website B)

**Paired t-test**: Used when same subjects measured twice (e.g., before vs after treatment)

<details>
<summary>Click to view solution</summary>

```python
# Independent t-test
t_stat, p_value = stats.ttest_ind(group_a, group_b)
print("T-statistic:", t_stat)
print("P-value:", p_value)

alpha = 0.05
if p_value < alpha:
    print("Statistically Significant")
else:
    print("Not Statistically Significant")
```
</details>

### Welch's t-test vs Permutation Test

<details>
<summary>Click to view solution</summary>

```python
# Welch's t-test (does not assume equal variances)
t_stat_w, t_pval_w = stats.ttest_ind(time_b, time_a, equal_var=False)

print(f"Permutation Test p-value: {p_val:.4f}")
print(f"Welch's t-test p-value:   {t_pval_w:.4f}")
print("\nThey should be very close, but the permutation test makes fewer assumptions about data shape.")
```
</details>

## Reflection

Questions:
1. Was result significant?
2. Could significance still be misleading?
3. Does significance imply usefulness?


---

# Section 8: Type I Error (False Positive)

## What is Type I Error?

Type I Error happens when we reject H₀ when H₀ is actually true.

Meaning: False positive — thinking new website works when it actually does not.

Probability of Type I Error: α = 0.05

### Intuition
If we run 100 experiments with 5% significance, we may get ~5 false positives purely by chance.

<details>
<summary>Click to view solution</summary>

```python
false_positives = 0

for _ in range(1000):
    a = np.random.normal(50, 10, 100)
    b = np.random.normal(50, 10, 100)
    _, p = stats.ttest_ind(a, b)
    if p < 0.05:
        false_positives += 1

print("False Positive Rate:", false_positives / 1000)
print("Expected: ~0.05")
```
</details>

## Reflection

Questions:
1. Did we observe roughly 5%?
2. Why do false positives happen?
3. Why is experimentation risky?



---

# Section 9: Type II Error (False Negative)

## What is Type II Error?

Type II Error happens when real effect exists but we fail to detect it.

Meaning: False negative — missing a genuinely better website.

### Causes of Type II Error
- Small sample size
- Noisy data
- Weak signal

<details>
<summary>Click to view solution</summary>

```python
small_a = np.random.normal(50, 10, 10)
small_b = np.random.normal(53, 10, 10)

_, p_small = stats.ttest_ind(small_a, small_b)
print("P-value with small sample:", p_small)
print("Note: Real effect exists (53 vs 50) but may not be detected with small n")
```
</details>

## Reflection

Questions:
1. Why might significance fail here?
2. Does no significance mean no effect?
3. Why are small datasets dangerous?


---

# Section 10: Statistical Power

## What is power?

Power measures ability to detect real effects. Higher power means lower chance of false negatives.

Power improves with:
- Larger sample size
- Stronger signal
- Lower noise

<details>
<summary>Click to view solution</summary>

```python
sample_sizes = [10, 50, 100, 500]
p_values_power = []

for n in sample_sizes:
    a = np.random.normal(50, 10, n)
    b = np.random.normal(53, 10, n)
    _, p = stats.ttest_ind(a, b)
    p_values_power.append(p)

power_df = pd.DataFrame({"Sample Size": sample_sizes, "P-value": p_values_power})
print(power_df)

plt.figure(figsize=(8, 5))
plt.plot(sample_sizes, p_values_power, marker="o")
plt.axhline(0.05, linestyle="--", label="Significance Threshold")
plt.legend()
plt.title("Sample Size vs P-value")
plt.xlabel("Sample Size")
plt.ylabel("P-value")
plt.grid(alpha=0.3)
plt.show()
```
</details>

## Reflection

Questions:
1. What happened as sample size increased?
2. Why does more data improve detection?
3. Why can tiny effects become significant?



---

# Section 11: Power via Simulation

## How many users do we need to detect a lift from 4% to 5% conversion?

<details>
<summary>Click to view solution</summary>

```python
def simulate_power(n_per_group, p_control, p_treatment, alpha=0.05, n_simulations=1000):
    significant_count = 0
    for _ in range(n_simulations):
        conv_a = np.random.binomial(1, p_control, n_per_group)
        conv_b = np.random.binomial(1, p_treatment, n_per_group)
        
        contingency = np.array([[sum(conv_a), n_per_group - sum(conv_a)],
                                [sum(conv_b), n_per_group - sum(conv_b)]])
        _, p_val, _, _ = chi2_contingency(contingency)
        
        if p_val < alpha:
            significant_count += 1
            
    return significant_count / n_simulations

# Test different sample sizes
sample_sizes_power = [1000, 3000, 5000, 10000, 15000]
powers = []

print("Simulating Power (detecting 4% -> 5% lift)...")
for n in sample_sizes_power:
    power = simulate_power(n, p_control=0.04, p_treatment=0.05, n_simulations=500)
    powers.append(power)
    print(f"  n={n:>5,} per group -> Power: {power:.1%}")

plt.figure(figsize=(8, 5))
plt.plot(sample_sizes_power, powers, 'o-', linewidth=2)
plt.axhline(0.80, color='red', linestyle='--', label='80% Power Target')
plt.xlabel('Sample Size per Group')
plt.ylabel('Statistical Power')
plt.title('Power Curve: Detecting 1% Absolute Lift in Conversion')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```
</details>


---

# Section 12: Multiple Testing Problem

## What happens if we test many hypotheses?

If we run 100 experiments with α = 0.05, expected false positives ≈ 5, even if nothing actually works.

The more tests you run, the more likely false discoveries become.

Examples:
- Testing 100 website changes
- Testing 100 ML features
- Testing 100 marketing campaigns

<details>
<summary>Click to view solution</summary>

```python
significant_results = 0

for _ in range(100):
    a = np.random.normal(50, 10, 100)
    b = np.random.normal(50, 10, 100)
    _, p = stats.ttest_ind(a, b)
    if p < 0.05:
        significant_results += 1

print("False Significant Results:", significant_results)
print("Expected ~5 by chance alone")
```
</details>

## Simulating Multiple Testing with 100 Tests

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)
n_tests = 100
p_values_multi = []

for _ in range(n_tests):
    a = np.random.normal(100, 15, 50)
    b = np.random.normal(100, 15, 50)
    _, p = stats.ttest_ind(a, b)
    p_values_multi.append(p)

p_values_multi = np.array(p_values_multi)
false_positives_multi = np.sum(p_values_multi < 0.05)
print(f"Out of {n_tests} tests with NO real effect, {false_positives_multi} were 'significant' (False Positives).")
```
</details>

## Reflection

Questions:
1. Did false positives appear?
2. Why is this dangerous?
3. How might companies fool themselves?


---

# Section 13: Multiple Testing Corrections

## Corrections for Multiple Testing

1. **Bonferroni**: Divide α by the number of tests. Very strict, high false negative rate.
2. **Benjamini-Hochberg (FDR)**: Controls the False Discovery Rate. Much more practical for data science.

<details>
<summary>Click to view solution</summary>

```python
# Apply corrections using statsmodels
reject_bonf, pvals_bonf, _, _ = multipletests(p_values_multi, alpha=0.05, method='bonferroni')
reject_fdr, pvals_fdr, _, _ = multipletests(p_values_multi, alpha=0.05, method='fdr_bh')

print(f"Significant tests after Bonferroni: {np.sum(reject_bonf)}")
print(f"Significant tests after FDR (Benjamini-Hochberg): {np.sum(reject_fdr)}")
print("\nBoth corrections should reduce false positives to near zero when H0 is true.")
```
</details>


---

# Section 14: P-hacking

## What is p-hacking?

P-hacking means searching for significance until something appears significant.

Examples:
- Trying many tests
- Removing inconvenient data
- Changing metrics repeatedly
- Stopping experiment early

Danger: Fake discoveries.

⚠️ A statistically significant result does not guarantee truth. Poor experimentation can still mislead.

<details>
<summary>Click to view solution</summary>

```python
p_values_hack = []

for _ in range(50):
    a = np.random.normal(50, 10, 100)
    b = np.random.normal(50, 10, 100)
    _, p = stats.ttest_ind(a, b)
    p_values_hack.append(p)

print("Minimum p-value from 50 tests with no true effect:", min(p_values_hack))
print("This demonstrates how easy it is to find 'significant' results by running many tests.")
```
</details>

## Reflection

Questions:
1. Did any small p-values appear?
2. Could these be fake discoveries?
3. Why should analysts avoid p-hacking?


---

# Section 15: ANOVA (Analysis of Variance)

## What is ANOVA?

ANOVA extends the t-test to 3 or more continuous groups. It tests if at least one group mean is different by comparing between-group variance to within-group variance (F-statistic).

<details>
<summary>Click to view solution</summary>

```python
groups = [group['time'].values for name, group in df_four_pages.groupby('page')]
f_stat, anova_p = f_oneway(*groups)

print(f"ANOVA F-statistic: {f_stat:.3f}")
print(f"P-value: {anova_p:.4f}")

if anova_p < 0.05:
    print("At least one page is significantly different.")
    print("(Requires post-hoc tests to find which one)")
```
</details>



---

# Section 16: Chi-Square Test

## What is Chi-Square Test?

Used for categorical/binary data (e.g., conversion rates across groups, contingency tables). Tests if observed counts differ from expected counts under independence.

<details>
<summary>Click to view solution</summary>

```python
contingency = pd.crosstab(df_price['price_group'], df_price['converted'])
display(contingency)

chi2, chi_p, dof, expected = chi2_contingency(contingency)
print(f"\nChi-Square statistic: {chi2:.3f}")
print(f"P-value: {chi_p:.4f}")

# Note: If expected counts are < 5, use Fisher's Exact Test
# odds_ratio, fisher_p = fisher_exact(contingency)
```
</details>



---

# Section 17: Multi-Arm Bandit Algorithms

## What are Multi-Arm Bandits?

Traditional A/B testing has a flaw: during the test, you waste traffic on the inferior variant. Multi-Arm Bandits solve this by dynamically allocating traffic, balancing Exploration (gathering data) and Exploitation (using the best variant).

### Epsilon-Greedy Strategy
- With probability ε: Explore (pick a random arm)
- With probability 1-ε: Exploit (pick the arm with highest historical reward)

<details>
<summary>Click to view solution</summary>

```python
class EpsilonGreedyBandit:
    def __init__(self, n_arms, epsilon=0.1):
        self.epsilon = epsilon
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        
    def select_arm(self):
        if np.random.random() < self.epsilon:
            return np.random.randint(len(self.values))
        return np.argmax(self.values)
        
    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] += (reward - self.values[arm]) / n

# True conversion rates of 3 arms (Arm 2 is the true winner)
true_rates = [0.05, 0.12, 0.08]
bandit = EpsilonGreedyBandit(n_arms=3, epsilon=0.1)

rewards_history = []
for step in range(2000):
    arm = bandit.select_arm()
    reward = np.random.binomial(1, true_rates[arm])
    bandit.update(arm, reward)
    rewards_history.append(reward)

# Results
print("Final Estimated Conversion Rates:")
for i, val in enumerate(bandit.values):
    print(f"  Arm {i}: {val:.3f} (True: {true_rates[i]}) | Pulled {int(bandit.counts[i])} times")

# Plot cumulative reward
plt.figure(figsize=(10, 5))
cumulative_avg = np.cumsum(rewards_history) / np.arange(1, len(rewards_history)+1)
plt.plot(cumulative_avg, label='Bandit Cumulative Conversion Rate')
plt.axhline(max(true_rates), color='red', linestyle='--', label='Optimal Arm True Rate (0.12)')
plt.axhline(np.mean(true_rates), color='gray', linestyle=':', label='Random Guess Average')
plt.xlabel('Steps (Users)')
plt.ylabel('Cumulative Conversion Rate')
plt.title('Epsilon-Greedy Bandit Learning Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```
</details>



---

# Section 18: Simulating Randomness

## Question

Can randomness alone create differences?

<details>
<summary>Click to view solution</summary>

```python
random_a = np.random.normal(50, 10, 1000)
random_b = np.random.normal(50, 10, 1000)

t_stat_rand, p_value_rand = stats.ttest_ind(random_a, random_b)
print("P-value:", p_value_rand)
print("Both groups drawn from same distribution - any 'difference' is purely random.")
```
</details>



---

# Section 19: Practical Business Example

## Airline Website Experiment

Question: Does new booking flow improve conversion?

- **Group A**: Current website (10% conversion)
- **Group B**: New website (11% conversion)

<details>
<summary>Click to view solution</summary>

```python
conversion_a = np.random.binomial(1, 0.10, 5000)
conversion_b = np.random.binomial(1, 0.11, 5000)

_, p_business = stats.ttest_ind(conversion_a, conversion_b)

print("P-value:", p_business)
print(f"Difference: {(conversion_b.mean() - conversion_a.mean())*100:.2f}%")
print("\nInterpretation:")
if p_business < 0.05:
    print("  Statistically significant")
else:
    print("  Not statistically significant")
print(f"  Absolute improvement: {(conversion_b.mean() - conversion_a.mean())*100:.2f} percentage points")
print("  Consider: Is this practically meaningful for the business?")
```
</details>



---

# Section 20: ML Relevance

## Why Chapter 3 matters in Machine Learning

Machine learning experiments are statistical experiments.

### Model Comparison
Did Model B improve accuracy, or was improvement random?

### A/B Testing
Used for recommender systems, ranking algorithms, ads systems, pricing models.

### False Positives
Danger: Thinking model improved when it actually did not.

### Multiple Testing
Trying many models can accidentally produce fake improvements.

### Statistical Thinking
Good ML engineers ask: "Is improvement reliable?" not "Did accuracy increase slightly?"

### ML Relevance Recap

| Concept | ML Application |
|---------|---------------|
| **Permutation Tests** | Evaluating if Model A is significantly better than Model B on a holdout set |
| **Multiple Testing** | Feature selection, hyperparameter search, avoiding p-hacking in model metrics |
| **Bandits** | Recommendation systems, dynamic pricing, AutoML pipeline selection |
| **Power Analysis** | Sizing validation/test sets so metric evaluations are statistically stable |

---

# Mini Project

## Goal

Design your own A/B test.

### Scenario Ideas
1. Airline booking flow
2. Recommendation system
3. New pricing strategy
4. Marketing email design
5. New ML model

### Tasks
1. Define H₀
2. Define H₁
3. Simulate data
4. Run t-test
5. Interpret p-value
6. Decide business action

---

# Interview Questions

1. **What is hypothesis testing?**
2. **What is null hypothesis?**
3. **What is alternative hypothesis?**
4. **What is p-value?**
5. **Does p-value measure probability H₀ is true?**
6. **Difference between statistical significance and business significance?**
7. **What is Type I error?**
8. **What is Type II error?**
9. **What is statistical power?**
10. **What is permutation testing?**
11. **Why are A/B tests important?**
12. **What is p-hacking?**
13. **How do you correct for multiple testing?**
14. **What is the difference between ANOVA and t-test?**
15. **When would you use a Chi-Square test?**

---

# Challenge Problems

1. Create A/B test simulator where effect size changes and observe p-value changes
2. Simulate 1000 false positives and calculate error rate
3. Compare small sample vs large sample for significance detection
4. Build permutation test vs t-test comparison
5. Simulate multiple testing problem with 500 experiments
6. Implement a more advanced bandit algorithm (UCB or Thompson Sampling)

---

# Chapter Summary

## Key Concepts Learned
- Statistical experiments and A/B testing
- Null hypothesis and alternative hypothesis
- Statistical significance and p-values
- Permutation tests (resampling approach)
- T-tests (classical and Welch's)
- Type I error (false positive)
- Type II error (false negative)
- Statistical power and sample size calculation
- Multiple testing problem and corrections (Bonferroni, FDR)
- P-hacking and its dangers
- ANOVA for multiple group comparison
- Chi-Square test for categorical data
- Multi-Arm Bandit algorithms (Epsilon-Greedy)

## Biggest takeaway

Statistics helps distinguish real signals from randomness. Small improvements are meaningless unless they are reliable.

---

# Progress Checklist

- [ ] Understood A/B testing
- [ ] Learned hypothesis testing
- [ ] Understood H₀ and H₁
- [ ] Practised p-values
- [ ] Built permutation tests from scratch
- [ ] Used t-tests (independent and Welch's)
- [ ] Understood Type I error
- [ ] Understood Type II error
- [ ] Learned statistical power
- [ ] Simulated power analysis
- [ ] Explored multiple testing
- [ ] Applied Bonferroni and FDR corrections
- [ ] Understood p-hacking
- [ ] Applied ANOVA for multiple groups
- [ ] Used Chi-Square for categorical data
- [ ] Implemented Epsilon-Greedy Bandit
- [ ] Connected concepts to ML
- [ ] Ready for Chapter 4

---

> **Pro Tip**: The permutation test function built in this notebook is a highly reusable snippet. Save it to your `utils/stats_helpers.py` file so you can import it in future projects when standard t-tests aren't appropriate!
